In [ ]:
library(Seurat)

In [ ]:
own <- readRDS("./data/macaque/classFiltSeurat_filt_allSubtypes.rds") 

# 1. Basic structure
own
head(rownames(own), 20)        # Gene names
head(colnames(own), 5)         # Cell barcodes
dim(own)                       # Genes x cells

# 2. Metadata columns
colnames(own@meta.data)

# 3. Major/fine class distribution
sort(table(own$cell_type_fine), decreasing = TRUE)

# 4. Find 5-HT related clusters
grep("5-?HT|serot|raphe", unique(own$cell_type_fine),
     value = TRUE, ignore.case = TRUE)

# 5. Predicted major class distribution (for coarse annotation during merge)
if ("maxPredClass" %in% colnames(own@meta.data)) {
  sort(table(own$maxPredClass), decreasing = TRUE)
} else {
  cat("No maxPredClass column\n")
}

In [ ]:
# 跨物种

In [ ]:
library(Seurat)

all_seu <- readRDS("./data/macaque/classFiltSeurat_filt_allSubtypes.rds")

# ---- 1. Object overview: assay, cell count, gene count, reductions ----
all_seu

# ---- 2. All metadata column names ----
colnames(all_seu@meta.data)

# ---- 3. Inspect each column (first 10 rows) ----
for (cc in colnames(all_seu@meta.data)) {
  cat("\n========== ", cc, " ==========\n")
  print(head(all_seu@meta.data[[cc]], 10))
}

# ---- 4. Value distribution of key columns ----
# Major class
table(all_seu$cell_class)

# Subclasses (may be long, check levels and counts)
length(levels(factor(all_seu$cell_type_new)))
sort(table(all_seu$cell_type_new), decreasing = TRUE)

# donor
table(all_seu$donor)

# ---- 5. Available assays / layers ----
Assays(all_seu)
Layers(all_seu[["RNA"]])
Layers(all_seu[["SCT"]])

# ---- 6. Available reductions and their dimensions ----
Reductions(all_seu)
sapply(Reductions(all_seu), function(r) dim(Embeddings(all_seu, r)))

# ---- 7. Clustering results ----
grep("snn_res", colnames(all_seu@meta.data), value = TRUE)
table(all_seu$seurat_clusters)

# ---- 8. Current Idents ----
levels(Idents(all_seu))

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "./output/"
save_dir <- "./output/cross_species/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)
## =========================================================================
## Part 1: Load annotated data, keep only 5-HT cells (no re-clustering, use cell_type_new directly)
## =========================================================================

own <- readRDS("./data/macaque/all_cells_reclustered.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

# Keep only 5-HT cells, use existing annotations directly
mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT subtype composition ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## Part 2: Cross-species integration (macaque × mouse, CCA dims 1:50, batch-balanced downsampling)
## =========================================================================

PAR <- list(
  variable_genes = 8000,    # ★ 8000 → 10000: approach one-to-one ortholog limit / ★ 8000 → 10000：逼近一对一同源基因上限
  npcs           = 100,
  cca_dims       = 1:80,     # ★ 60 → 80: expand anchor-related space / ★ 60 → 80：锚点相关空间再放宽
  nb_dims        = 1:30,     
  resolution     = 0.1,      # [PARAM] resolution=0.1 is VERY LOW; try 0.3-0.8 / 不变（再强调：res 不影响整合，只影响聚类粒度）
  min_cells      = 100,
  k_anchor       = 5        # ★ New: anchor k value, default 5 → 10 / ★ 新增：锚点 k 值，默认 5 → 10
)

## ---- 2.1 Read data + clean up ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("./data/mouse_ref/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 Ortholog gene conversion (unchanged) ----
test <- read.table('./data/mouse_ref/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("One-to-one orthologs: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 Merge small donors + downsample each batch (unchanged) ----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                          table(mouseHomoSeurat$batch)))
message("Downsample each batch to ", target_per_batch)

downsample_by_batch <- function(obj, target_per_batch) {
  ct <- obj$clusterName; bt <- obj$batch
  cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
    n_target <- min(length(idx), target_per_batch)
    tab <- table(ct[idx])
    n_per <- round(tab / sum(tab) * n_target)
    n_per <- pmin(n_per, tab)
    n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
    unlist(lapply(names(n_per), function(x) {
      cells_x <- colnames(obj)[idx][ct[idx] == x]
      sample(cells_x, min(n_per[x], length(cells_x)))
    }))
  }))
  sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
}
macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                            cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
mouseHomoSeurat   <- subset(mouseHomoSeurat,
                            cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

## ---- 2.4 Split + merge ----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA  ----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 Clustering + UMAP (unchanged) ----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)  
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)  
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 Visualization ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# ★ Use existing annotations: macaque = mac_5-HT_x (prefix to distinguish from mouse), mouse = 5-HT_x 
mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("Cells with missing labels:", n_missing, "/", ncol(classFiltSeurat), "\n")  
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (macaque = annotated 5-HT)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap heatmap (rows = mac_5-HT_x, cols = mouse 5-HT_x) ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap matrix:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(out_dir, "overlap_cca_dims50_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_mac5HT.csv"))
cat("Overlap heatmap saved\n")

In [ ]:
## ---- 2.7 Visualization (pt.size further increased) ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE,
              pt.size = 1.5
            ) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)

In [ ]:
## ---- 2.9 Save ----
saveRDS(classFiltSeurat, paste0(save_dir, "crossSpecies_CCA_dims50_mac5HT.rds"))
writeLines(capture.output(sessionInfo()),
           paste0(save_dir, "sessionInfo_CCA_dims50_mac5HT.txt"))
message("All done")

In [ ]:
# Sankey

In [ ]:
save_dir <- "./output/cross_species/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
## =========================================================================
## 2.9 Sankey: macaque original subtypes → (kNN mapping) → mouse subtypes
## =========================================================================
suppressMessages({ library(ggalluvial); library(RANN); library(reshape2) })

# ---- 1. kNN label transfer in integrated.cca space ----
mac_cells <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mus_cells <- colnames(classFiltSeurat)[classFiltSeurat$species == "mouse"]

emb <- Embeddings(classFiltSeurat, "integrated.cca")[, 1:50]
emb_mac <- emb[mac_cells, , drop = FALSE]
emb_mus <- emb[mus_cells, , drop = FALSE]

mus_label <- as.character(classFiltSeurat$subtype_recl[mus_cells])

k <- 10
nn <- nn2(emb_mus, emb_mac, k = k)          # 10 nearest mouse cells per macaque cell
pred_mouse <- vapply(seq_len(nrow(nn$nn.idx)), function(i) {
  labs <- mus_label[nn$nn.idx[i, ]]
  names(sort(table(labs), decreasing = TRUE))[1]
}, character(1))

# Each macaque cell: original subtype + predicted mouse subtype
mac_lab <- as.character(classFiltSeurat$subtype_recl[mac_cells])

# ---- 2. Summarize as macaque × mouse count matrix (melt style) ----
NV <- table(mac_lab, pred_mouse)

# Sort by trailing number: mac_5-HT_0..10, 5-HT_0..n (avoid lexicographic 10 before 2)
sort_by_num <- function(x) {
  x[order(as.numeric(sub(".*_(\\d+)$", "\\1", x)))]
}
NV <- NV[sort_by_num(rownames(NV)), sort_by_num(colnames(NV))]

mac_types <- rownames(NV)
mus_types <- colnames(NV)

df <- melt(NV, varnames = c("macaque", "mouse"), value.name = "n")
df$macaque <- factor(df$macaque, levels = mac_types)
df$mouse   <- factor(df$mouse,   levels = mus_types)

# Mapping consistency (print as supplementary table)
map_rate <- df %>%
  group_by(macaque) %>%
  summarise(top_pred = mouse[which.max(n)],
            frac = max(n) / sum(n), .groups = "drop")
print(map_rate)
write.csv(map_rate, paste0(save_dir, "mac_to_mouse_knn_mapping.csv"),
          row.names = FALSE)

# ---- 3. Full Sankey diagram (MetaNeighbor style) ----
p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = n)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) +
  scale_x_continuous(breaks = 1:2,
                     labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque → mouse 5-HT subtype mapping (kNN in CCA space)")

print(p_sankey)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.pdf"),
       p_sankey, width = 14, height = 7)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.png"),
       p_sankey, width = 14, height = 7, dpi = 300)

# ---- 4. Show only strong correspondences (>30% of cells), cleaner plot ----
df_strong <- df %>%
  group_by(macaque) %>%
  mutate(frac = n / sum(n)) %>%
  ungroup() %>%
  subset(frac > 0.3)

if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = n)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3.5) +
    scale_x_continuous(breaks = 1:2,
                       labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque → mouse (kNN mapping, >30% of subtype)")
  print(p_sankey_strong)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.png"),
         p_sankey_strong, width = 12, height = 7, dpi = 300)
}

cat("Sankey diagrams saved\n")

In [ ]:
# ---- 3. Full Sankey diagram ----
p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = n)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  scale_x_continuous(breaks = 1:2,
                     labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque → mouse 5-HT subtype mapping (kNN in CCA space)")

print(p_sankey)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.pdf"),
       p_sankey, width = 14, height = 7)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.png"),
       p_sankey, width = 14, height = 7, dpi = 300)

# ---- 4. Strong correspondence Sankey ----
df_strong <- df %>%
  group_by(macaque) %>%
  mutate(frac = n / sum(n)) %>%
  ungroup() %>%
  subset(frac > 0.3)

if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = n)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    scale_x_continuous(breaks = 1:2,
                       labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque → mouse (kNN mapping, >30% of subtype)")
  print(p_sankey_strong)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.png"),
         p_sankey_strong, width = 12, height = 7, dpi = 300)
}

In [ ]:
# Gene plot

In [ ]:
install.packages("ggvenn")

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(ggvenn)
})

save_dir <- "./output/cross_species/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. Data preparation (unchanged) ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

keep_pat <- "^5-HT_[0-9]+$"
is_bad <- !grepl(keep_pat, classFiltSeurat$sub_show)
if (any(is_bad)) {
  cat("Removed non-numbered subtype cells:", sum(is_bad), fill = TRUE)
  classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!is_bad])
}

cap_first <- function(x) paste0(toupper(substr(x, 1, 1)), substr(x, 2, nchar(x)))

# ---------- 1. Find subtype markers within each species (unchanged) ----------
find_mk <- function(obj) {
  Idents(obj) <- "sub_show"
  mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                       min.pct = 0.1, logfc.threshold = 1,
                       test.use = "wilcox", verbose = FALSE)
  if (is.null(mk) || nrow(mk) == 0 || !"p_val_adj" %in% colnames(mk)) {
    mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                         min.pct = 0.05, logfc.threshold = 0.5,
                         test.use = "wilcox", verbose = FALSE)
  }
  stopifnot(!is.null(mk), nrow(mk) > 0, "p_val_adj" %in% colnames(mk))
  mk %>% filter(p_val_adj < 0.01) %>%      # p threshold 0.01
    mutate(cluster = as.character(cluster))
}

species_levels <- sort(unique(as.character(classFiltSeurat$species)))
species_levels <- c(intersect(c("macaque", "marmoset", "mouse"), species_levels),
                    setdiff(species_levels, c("macaque", "marmoset", "mouse")))
n_sp <- length(species_levels)

obj_list <- lapply(species_levels, function(sp)
  subset(classFiltSeurat, subset = species == sp))
names(obj_list) <- species_levels
mk_list <- lapply(obj_list, find_mk)

# ---------- 2. Venn diagram: marker gene sets by species ----------
gene_sets <- lapply(mk_list, function(mk) unique(mk$gene))
names(gene_sets) <- cap_first(species_levels)

all_genes <- unique(unlist(gene_sets))          # First define universal set
venn_df <- as.data.frame(                       # Then build 0/1 matrix
  sapply(gene_sets, function(g) all_genes %in% g)
)
rownames(venn_df) <- all_genes

## Calculate gene list for each region
region_genes <- list()
for (i in seq_len(nrow(venn_df))) {
  reg <- paste(names(venn_df)[as.logical(venn_df[i, ])], collapse = "&")
  region_genes[[reg]] <- c(region_genes[[reg]], rownames(venn_df)[i])
}

## Venn diagram main body
p_venn <- ggplot(venn_df) +
  geom_venn(aes(A = .data[[names(venn_df)[1]]],
                B = .data[[names(venn_df)[2]]],
                C = if (ncol(venn_df) >= 3) .data[[names(venn_df)[3]]] else NULL),
            fill_color = c("#EB6368", "#E8A33D", "#82ADD0")[seq_len(ncol(venn_df))],
            fill_alpha = 0.45, stroke_color = "white", stroke_size = 0.5,
            text_size = 5, set_name_size = 5) +
  coord_fixed() +
  theme_void()

ggsave(paste0(save_dir, "Fig_markerVennyouzi.pdf"), p_venn, width = 8, height = 8)
ggsave(paste0(save_dir, "Fig_markerVennyouzi.png"), p_venn, width = 8, height = 8, dpi = 300)

# ---------- 3. Export gene lists by region ----------
region_summary <- data.frame(
  region = names(region_genes),
  n_genes = lengths(region_genes)
)
write.csv(region_summary, paste0(save_dir, "Table_markerVenn_regionsyouzi.csv"),
          row.names = FALSE)
for (reg in names(region_genes)) {
  safe_reg <- gsub("[^A-Za-z0-9]+", "_", reg)
  write.csv(data.frame(gene = region_genes[[reg]]),
            paste0(save_dir, "Genes_", safe_reg, ".csv"), row.names = FALSE)
}

cat("Venn diagram saved to", save_dir, fill = TRUE)
print(region_summary)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(patchwork)
})

save_dir <- "./output/cross_species/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. Data preparation ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

# ---------- 1. Cross-species DEG (macaque vs mouse, all 5-HT cells) ----------
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht   <- subset(classFiltSeurat, cells = cells_5ht)
Idents(obj_5ht) <- "species"

deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)

# ---------- 2. Top 30 up / down ----------
top_up <- deg %>% filter(avg_log2FC > 0) %>%
  arrange(desc(avg_log2FC)) %>% head(200) %>% rownames()
top_dn <- deg %>% filter(avg_log2FC < 0) %>%
  arrange(avg_log2FC) %>% head(200) %>% rownames()

# ---------- 3. Get real expression (sample 3000 cells per species for faster plotting) ----------
set.seed(1)
expr <- FetchData(obj_5ht, vars = c(top_up, top_dn, "species"))
expr_s <- expr %>%
  group_by(species) %>%
  slice_sample(n = 3000) %>%
  ungroup()

to_long <- function(genes) {
  expr_s %>%
    select(species, all_of(genes)) %>%
    pivot_longer(-species, names_to = "gene", values_to = "expr") %>%
    mutate(gene   = factor(gene, levels = genes),
           species = factor(species, levels = c("mouse", "macaque")))  # Left mouse, right macaque
}
long_up <- to_long(top_up)
long_dn <- to_long(top_dn)

# ---------- 4. Plot ----------
sp_fill <- c("mouse" = "#82ADD0", "macaque" = "#EB6368")

vln_panel <- function(df, title) {
  ggplot(df, aes(species, expr, fill = species)) +
    geom_violin(scale = "width", linewidth = 0.1, color = NA, na.rm = TRUE) +
    geom_boxplot(width = 0.12, outlier.shape = NA, alpha = 0.6,
                 linewidth = 0.2, color = "grey20") +
    scale_fill_manual(values = sp_fill) +
    scale_x_discrete(drop = FALSE) +
    facet_wrap(~ gene, scales = "free_y", ncol = 6) +
    labs(title = title, x = NULL, y = "Normalized expression") +
    theme_bw(base_size = 9) +
    theme(legend.position = "none",
          strip.text = element_text(face = "italic", size = 8),
          axis.text.x  = element_text(size = 8),
          plot.title   = element_text(face = "bold", size = 12))
}

p_up <- vln_panel(long_up, "Top 200 up in macaque (vs mouse)")
p_dn <- vln_panel(long_dn, "Top 200 down in macaque (up in mouse)")

pdf(paste0(save_dir, "Fig_5HT_top50_up_down_violin.pdf"), width = 12, height = 18)
print(p_up / p_dn + plot_layout(heights = c(1, 1)))
dev.off()

png(paste0(save_dir, "Fig_5HT_top200_up_down_violin.png"),
    width = 12, height = 18, units = "in", res = 300)
print(p_up / p_dn + plot_layout(heights = c(1, 1)))
dev.off()
cat("Violin plot saved to", save_dir, fill = TRUE)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(patchwork)
})

save_dir <- "./output/cross_species/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

GENE <- "NXPH2"     # Change gene here

# ---------- 0. Data preparation ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht   <- subset(classFiltSeurat, cells = cells_5ht)
stopifnot(GENE %in% rownames(obj_5ht))

sort_by_num <- function(x) {
  n <- suppressWarnings(as.numeric(sub(".*_(\\d+)$", "\\1", x)))
  x[order(n, na.last = TRUE)]
}

# ---------- 1. Get expression (data = log-normalized) and raw count ----------
df <- FetchData(obj_5ht, vars = c(GENE, "species", "sub_show"))
colnames(df)[1] <- "expr"
cnt <- FetchData(obj_5ht, vars = GENE, layer = "counts")
df$count <- cnt[rownames(df), 1]
df$species  <- factor(as.character(df$species), levels = c("macaque", "mouse"))
df$sub_show <- factor(as.character(df$sub_show),
                      levels = sort_by_num(unique(as.character(df$sub_show))))
df$express  <- df$count > 0

# ---------- 2. Violin plot: each species' own subtypes ----------
sp_fill <- c("macaque" = "#EB6368", "mouse" = "#82ADD0")

p_vln <- ggplot(df, aes(sub_show, expr, fill = species)) +
  geom_violin(scale = "width", linewidth = 0.1, color = NA) +
  scale_fill_manual(values = sp_fill) +
  facet_wrap(~ species, nrow = 1, scales = "free") +
  labs(title = paste0(GENE, " expression across 5-HT subclasses"),
       subtitle = "Raw UMI counts", x = NULL, y = "Raw count") +
  theme_classic(base_size = 11) +
  theme(legend.position = "none",
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1))

# ---------- 3. Expression proportion: fraction of expressing cells per subtype (cross-species) ----------
pct_df <- df %>%
  group_by(species, sub_show) %>%
  summarise(pct = mean(express), n = n(), .groups = "drop")

## Fisher exact test for each shared subtype
subs_mac  <- sort_by_num(unique(as.character(df$sub_show[df$species == "macaque"])))
subs_mus  <- sort_by_num(unique(as.character(df$sub_show[df$species == "mouse"])))
subs_both <- intersect(subs_mac, subs_mus)

sub_stat <- lapply(subs_both, function(s) {
  d <- df[as.character(df$sub_show) == s, ]
  if (n_distinct(d$species) < 2) return(NULL)
  tb <- table(d$species, d$express)
  if (nrow(tb) < 2 || ncol(tb) < 2) return(NULL)
  ft <- fisher.test(tb)
  data.frame(sub_show = s,
             pct_mac = mean(d$express[d$species == "macaque"]),
             pct_mus = mean(d$express[d$species == "mouse"]),
             p_val = ft$p.value)
})
sub_stat <- bind_rows(sub_stat)
if (nrow(sub_stat) > 0) {
  sub_stat$p_adj <- p.adjust(sub_stat$p_val, method = "BH")
  sub_stat$star <- cut(sub_stat$p_adj,
                       breaks = c(-1, 0.001, 0.01, 0.05, 1),
                       labels = c("***", "**", "*", "ns"))
}

lab_df <- pct_df %>%
  left_join(sub_stat %>% select(sub_show, star), by = "sub_show") %>%
  mutate(lab = paste0(round(pct * 100), "%"))

p_pct <- ggplot(pct_df, aes(sub_show, pct, fill = species)) +
  geom_col(position = position_dodge(0.85), width = 0.75, color = NA) +
  geom_text(data = lab_df,
            aes(label = lab), position = position_dodge(0.85),
            vjust = -0.4, size = 3) +
  geom_text(data = sub_stat,
            aes(x = sub_show, y = max(pct_mac, pct_mus) + 0.12,
                label = star), inherit.aes = FALSE, size = 4) +
  scale_fill_manual(values = sp_fill) +
  scale_y_continuous(labels = scales::percent, expand = expansion(mult = c(0, 0.15))) +
  labs(title = paste0(GENE, ": fraction of expressing cells"),
       subtitle = "Fisher exact test, BH-adjusted", x = NULL, y = "% of cells") +
  theme_classic(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# ---------- 4. Overall proportion: cross-species at all 5-HT cells level ----------
overall <- df %>%
  group_by(species) %>%
  summarise(pct = mean(express), n = n(), .groups = "drop")
tab_all <- table(df$species, df$express)
ft_all  <- fisher.test(tab_all)
p_all   <- chisq.test(tab_all)$p.value

p_ov <- ggplot(overall, aes(species, pct, fill = species)) +
  geom_col(width = 0.6, color = NA) +
  geom_text(aes(label = paste0(round(pct * 100, 1), "%\n(n=", n, ")")),
            vjust = -0.3, size = 3.5) +
  scale_fill_manual(values = sp_fill) +
  scale_y_continuous(labels = scales::percent, expand = expansion(mult = c(0, 0.15))) +
  labs(title = paste0(GENE, ": overall expression in 5-HT cells"),
       subtitle = paste0("Fisher p = ", signif(ft_all$p.value, 3),
                         " | chi-sq p = ", signif(p_all, 3)),
       x = NULL, y = "% of 5-HT cells") +
  theme_classic(base_size = 11) +
  theme(legend.position = "none")

# ---------- 5. Plot + statistics table ----------
p_all_fig <- (p_vln / p_pct) | p_ov
p_all_fig <- p_all_fig + plot_layout(widths = c(3, 1))

ggsave(paste0(save_dir, "Fig_", GENE, "_subclass_violin_pct.pdf"),
       p_all_fig, width = 14, height = 10)
ggsave(paste0(save_dir, "Fig_", GENE, "_subclass_violin_pct.png"),
       p_all_fig, width = 14, height = 10, dpi = 300)

write.csv(sub_stat, paste0(save_dir, "Table_", GENE, "_pct_by_subclass.csv"),
          row.names = FALSE)
write.csv(overall, paste0(save_dir, "Table_", GENE, "_pct_overall.csv"),
          row.names = FALSE)

cat("==== ", GENE, " Overall expression proportion ====", fill = TRUE)
print(overall)
cat("Fisher p =", ft_all$p.value, " | chi-sq p =", p_all, fill = TRUE)
cat("Figures and tables saved to", save_dir, fill = TRUE)

In [ ]:
# ---- 4. NXPH2 expression violin plot: macaque vs mouse ----
## Extract NXPH2 expression data
df_nxph2 <- FetchData(obj_5ht, vars = c("NXPH2", "species"))
df_nxph2$species <- factor(df_nxph2$species, levels = c("macaque", "mouse"))

p_nx <- ggplot(df_nxph2, aes(x = species, y = NXPH2, fill = species)) +
  ## Pure violin: no boxplot, no median line, no jitter points
  geom_violin(scale = "width", trim = TRUE, adjust = 1,
              linewidth = 0.3, color = "black") +
  scale_fill_manual(values = c(macaque = "#EB6368", mouse = "#82ADD0")) +
  labs(x = NULL, y = "NXPH2 expression", fill = NULL) +
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text        = element_text(family = ft, size = 8),
    axis.text   = element_text(family = ft, size = 8),
    axis.title  = element_text(family = ft, size = 8),
    legend.text = element_text(family = ft, size = 8),
    legend.position = "none",          # Colors already correspond to x-axis, no legend needed
    plot.margin = margin(6, 10, 6, 6)
  )

ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.pdf"),
       p_nx, width = 4, height = 4.5, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.png"),
       p_nx, width = 4, height = 4.5, units = "cm", dpi = 600)

print(p_nx)

In [ ]:
# go

In [ ]:
library(dplyr)
library(ggplot2)
save_dir <- "./output/cross_species/"
## ============================================================
## Diagnosis 1: genes passing filter per subtype under different threshold combinations
## ============================================================
threshold_grid <- expand.grid(
  FC  = c(0.25, 0.5, 0.75, 1.0),
  PCT = c(0.1, 0.2, 0.3, 0.5),
  stringsAsFactors = FALSE
)

diag_pass <- lapply(split(threshold_grid, seq_len(nrow(threshold_grid))), function(th) {
  mk_mac %>%
    filter(p_val_adj < 0.05, avg_log2FC > th$FC, pct.1 > th$PCT) %>%
    count(cluster, name = "n_pass") %>%
    mutate(FC = th$FC, PCT = th$PCT)
}) %>% bind_rows()

# Wide table: rows = subtypes, cols = threshold combinations
diag_wide <- diag_pass %>%
  mutate(combo = paste0("FC>", FC, "_pct>", PCT)) %>%
  select(cluster, combo, n_pass) %>%
  pivot_wider(names_from = combo, values_from = n_pass, values_fill = 0) %>%
  arrange(cluster)

print(diag_wide, n = Inf)
write.csv(diag_wide, paste0(save_dir, "Table_threshold_diagnostic.csv"),
          row.names = FALSE)

## ============================================================
## Diagnosis 2: avg_log2FC distribution per subtype (determine FC cutoff)
## ============================================================
fc_dist <- mk_mac %>%
  filter(p_val_adj < 0.05) %>%
  group_by(cluster) %>%
  summarise(
    n_sig      = n(),
    fc_median  = round(median(avg_log2FC), 2),
    fc_top10   = round(sort(avg_log2FC, decreasing = TRUE)[min(10, n())], 2),
    fc_top30   = round(sort(avg_log2FC, decreasing = TRUE)[min(30, n())], 2),
    fc_top50   = round(sort(avg_log2FC, decreasing = TRUE)[min(50, n())], 2),
    .groups = "drop"
  ) %>%
  arrange(cluster)

print(fc_dist, n = Inf)
# fc_top10/30/50 = fold change of gene ranked 10/30/50 by FC in this subtype
# If fc_top50 is only 0.4, many genes in this subtype's top 50 have FC < 0.5

## ============================================================
## Diagnosis 3: What TOP_N? Check cumulative distribution after FC>0.5 & pct>0.3
## ============================================================
n_after <- mk_mac %>%
  filter(p_val_adj < 0.05, avg_log2FC > 0.5, pct.1 > 0.3) %>%
  count(cluster, name = "n_after_filter") %>%
  arrange(cluster)

print(n_after, n = Inf)
hist(n_after$n_after_filter, breaks = 20,
     main = "Distribution of genes per subtype passing FC>0.5 & pct>0.3",
     xlab = "Gene count")

## ============================================================
## Diagnosis 4 (quick method): find threshold neither too strict nor too lenient
# ============================================================
# Target: most subtypes retain 30~100 genes after filtering (GO sweet spot)
target_check <- diag_pass %>%
  group_by(FC, PCT) %>%
  summarise(
    n_subtype     = n(),
    min_genes     = min(n_pass),
    median_genes  = round(median(n_pass)),
    n_below_30    = sum(n_pass < 30),    # Too few remaining, GO unstable subtypes
    n_above_100   = sum(n_pass > 100),   # Too many remaining, TOP_N truncation wasted
    .groups = "drop"
  ) %>%
  arrange(desc(n_below_30 == 0), median_genes)   # Prioritize combinations where no subtype is below 30

print(target_check, n = Inf)

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")
BiocManager::install("org.Mm.eg.db")

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db); library(org.Mm.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})
save_dir <- "./output/cross_species/"

## ============================================================
## 0. Define two analysis units (change here to switch gene sets)
## ============================================================
analyses <- list(
  list(label    = "macaque_specific",
       genes    = panel %>% filter(status == "Macaque-specific") %>% pull(gene),
       bg       = unique(mk_mac$gene),        # All genes tested by macaque FindAllMarkers
       OrgDb    = org.Hs.eg.db),
  list(label    = "mouse_specific",
       genes    = panel %>% filter(status == "Mouse-specific") %>% pull(gene),
       bg       = unique(mk_mus$gene),        # mouse tested genes
       OrgDb    = org.Mm.eg.db)
)

## ============================================================
## 1. Enrichment + plotting function
## ============================================================
run_one <- function(gene_set, bg_genes, OrgDb, label) {
  cat("\n========== ", label, " ==========\n")
  cat("Gene set size:", length(gene_set), "  Background set:", length(bg_genes), "\n")

  # Key: use same bitr conversion for gene set and background for comparable mapping rates
  gene_map <- bitr(gene_set, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = OrgDb)
  bg_map   <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = OrgDb)

  cat("Gene set mapping rate:", nrow(gene_map), "/", length(gene_set),
      " (", round(100 * nrow(gene_map) / length(gene_set), 1), "%)\n", sep = "")
  cat("Background mapping rate:", nrow(bg_map), "/", length(bg_genes),
      " (", round(100 * nrow(bg_map) / length(bg_genes), 1), "%)\n", sep = "")

  ego_bp <- enrichGO(gene          = gene_map$ENTREZID,
                     universe      = bg_map$ENTREZID,
                     OrgDb         = OrgDb,
                     keyType       = "ENTREZID",
                     ont           = "BP",
                     pAdjustMethod = "BH",
                     pvalueCutoff  = 0.05,
                     qvalueCutoff  = 0.2,
                     readable      = TRUE)

  df <- as.data.frame(ego_bp)
  cat("Enriched to ", nrow(df), " BP pathways\n")

  write.csv(df, paste0(save_dir, "GO_BP_", label, ".csv"), row.names = FALSE)

  if (nrow(df) > 0) {
    p_dot <- dotplot(ego_bp, showCategory = 20, font.size = 9) +
      ggtitle(paste0("GO BP: ", label))
    ggsave(paste0(save_dir, "GO_BP_", label, "_dotplot.pdf"),
           p_dot, width = 8, height = 7)
    ggsave(paste0(save_dir, "GO_BP_", label, "_dotplot.png"),
           p_dot, width = 8, height = 7, dpi = 300)

    p_bar <- barplot(ego_bp, showCategory = 20, font.size = 9) +
      ggtitle(paste0("GO BP: ", label))
    ggsave(paste0(save_dir, "GO_BP_", label, "_barplot.pdf"),
           p_bar, width = 8, height = 7)

    p_net <- cnetplot(ego_bp, showCategory = 8, circular = FALSE,
                      colorEdge = TRUE, node_label = "gene",
                      cex_label_category = 0.9, cex_label_gene = 0.7)
    ggsave(paste0(save_dir, "GO_BP_", label, "_cnetplot.pdf"),
           p_net, width = 11, height = 9)

    ego_s <- pairwise_termsim(ego_bp)
    p_tree <- treeplot(ego_s, showCategory = 20)
    ggsave(paste0(save_dir, "GO_BP_", label, "_treeplot.pdf"),
           p_tree, width = 9, height = 7)

    print(p_dot)
  }
  invisible(df)
}

## ============================================================
## 2. Run two species-specific sets
## ============================================================
go_results <- lapply(analyses, function(a)
  run_one(a$genes, a$bg, a$OrgDb, a$label))
names(go_results) <- sapply(analyses, `[[`, "label")

In [ ]:
suppressMessages({ library(ggplot2); library(dplyr) })

save_dir <- "./output/cross_species/"

## ============================================================
## Read results + filter (consistent with previous non-raphe blacklist)
## ============================================================
go_mac <- read.csv(paste0(save_dir, "GO_BP_macaque_specific.csv"))

drop_terms <- c("ossification", "osteoblast differentiation",
                "epithelial cell proliferation",
                "positive regulation of epithelial cell proliferation",
                "morphogenesis of a branching epithelium",
                "muscle cell proliferation",
                "response to lipopolysaccharide",
                "response to molecule of bacterial origin")

go_show <- go_mac %>% filter(!Description %in% drop_terms)

TOP_N <- 10

plot_df <- go_show %>%
  slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
  mutate(neglogP = -log10(p.adjust),
         label   = paste0(Description, " (", ID, ")"),
         # Most significant terms on top
         label   = factor(label, levels = rev(label)))

## ============================================================
## Bar plot
## ============================================================
p_bar <- ggplot(plot_df, aes(x = neglogP, y = label, fill = neglogP)) +
  geom_bar(stat = "identity", width = 0.7, color = NA) +
  scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
                       midpoint = max(plot_df$neglogP) / 2,
                       name = "-log10(Padj)") +
  scale_x_continuous(expand = expansion(mult = c(0, 0.1))) +
  labs(title = "Top GO Biological Processes",
       x = "-log10(Padj)", y = NULL) +
  theme_classic(base_size = 10) +
  theme(plot.title  = element_text(hjust = 0.5, face = "bold", size = 12),
        axis.text.y = element_text(size = 9, color = "black"),
        axis.text.x = element_text(size = 9, color = "black"),
        axis.line.y = element_blank(),
        axis.ticks.y = element_blank(),
        legend.position = "right",
        legend.title = element_text(size = 9),
        legend.text  = element_text(size = 8),
        legend.key.size = unit(0.4, "cm"))

print(p_bar)

ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_custom.png"),
       p_bar, width = 8, height = 5.5, dpi = 300)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_custom.pdf"),
       p_bar, width = 8, height = 5.5)

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})

save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/跨物种1011/"

# ---- 1. 基因集和背景集 ----
mac_genes <- gene_table %>% filter(conservation == "Macaque-specific") %>% pull(gene)
bg_genes  <- rownames(mk)          # FindMarkers 检验过的全部基因做背景，比全基因组更严谨
cat("macaque-specific:", length(mac_genes), " background:", length(bg_genes), "\n")

# ---- 2. GO 富集（BP 为主，三个本体可都跑）----
run_go <- function(ont) {
  enrichGO(gene          = mac_genes,
           universe      = bg_genes,
           OrgDb         = org.Hs.eg.db,
           keyType       = "SYMBOL",
           ont           = ont,
           pAdjustMethod = "BH",
           pvalueCutoff  = 0.05,
           qvalueCutoff  = 0.2,
           readable      = TRUE)
}

ego_bp <- run_go("BP")
ego_cc <- run_go("CC")
ego_mf <- run_go("MF")

# 看看映射率（猴符号 → 人源 Entrez 会丢一部分，属正常）
cat("BP 富集到 ", nrow(as.data.frame(ego_bp)), " 条通路\n")

# ---- 3. 保存结果表 ----
write.csv(as.data.frame(ego_bp), paste0(save_dir, "GO_BP_macaque_specific.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_cc), paste0(save_dir, "GO_CC_macaque_specific.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_mf), paste0(save_dir, "GO_MF_macaque_specific.csv"), row.names = FALSE)

# ---- 4. 出图 ----
# 条形图
p_bar <- barplot(ego_bp, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot.pdf"), p_bar, width = 8, height = 7)

# 气泡图
p_dot <- dotplot(ego_bp, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot.pdf"), p_dot, width = 8, height = 7)

# 基因-通路网络图（最直观，哪些基因推富集）
p_net <- cnetplot(ego_bp, showCategory = 8, circular = FALSE,
                  colorEdge = TRUE, node_label = "gene",
                  cex_label_category = 0.9, cex_label_gene = 0.7)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_cnetplot.pdf"), p_net, width = 11, height = 9)

# 聚类树状图（简化冗余通路）
ego_bp_s <- pairwise_termsim(ego_bp)
p_tree <- treeplot(ego_bp_s, showCategory = 20)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot.pdf"), p_tree, width = 9, height = 7)

print(p_dot)

In [ ]:
save_dir <- "/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/跨物种2/"
# ---- 4. 出图（先剔除非 raphe/污染相关术语）----
# 要删的术语关键词：纤毛、感觉/视觉、精子、附肢/肢体、免疫/炎症
drop_kw <- paste(c(
  "cilium", "flagell", "sperm",
  "sensory perception", "visual perception", "detection of",
  "phototransduction", "appendage", "limb",
  "immune", "inflammatory", "leukocyte", "lymphocyte",
  "defense response", "complement", "cytokine"
), collapse = "|")

res <- ego_bp@result
cat("剔除前:", nrow(res), "条\n")
keep <- !grepl(drop_kw, res$Description, ignore.case = TRUE)
ego_bp_filt <- ego_bp
ego_bp_filt@result <- res[keep, ]
cat("剔除后:", nrow(ego_bp_filt@result), "条\n")
print(ego_bp_filt@result$Description)

# 保存清洗后的表
write.csv(as.data.frame(ego_bp_filt),
          paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
          row.names = FALSE)

# 气泡图（清洗版）
p_dot <- dotplot(ego_bp_filt, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot_clean.pdf"),
       p_dot, width = 8, height = 6)
print(p_dot)

# 条形图（清洗版）
p_bar <- barplot(ego_bp_filt, showCategory = 15, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_clean.pdf"),
       p_bar, width = 8, height = 6)

# 树状图（清洗版，看主题聚类）
ego_filt_s <- pairwise_termsim(ego_bp_filt)
p_tree <- treeplot(ego_filt_s, showCategory = 15)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot_clean.pdf"),
       p_tree, width = 9, height = 6)

In [ ]:
# metaneighbor

In [ ]:
formals(MetaNeighborUS)

In [ ]:
suppressMessages(library(MetaNeighbor))
suppressMessages(library(SummarizedExperiment))

## ---- 1. Prepare input ----
expr_mat <- GetAssayData(classFiltSeurat[["SCT"]], slot = "data")
use_genes <- intersect(VariableFeatures(classFiltSeurat), rownames(expr_mat))
expr_mat <- as.matrix(expr_mat[use_genes, ])

keep_cells <- colnames(classFiltSeurat)[classFiltSeurat$subtype_recl != "Unassigned"]
meta_nn <- data.frame(
  cell_id    = keep_cells,
  study_id   = as.character(classFiltSeurat$species[keep_cells]),
  cell_label = as.character(classFiltSeurat$subtype_recl[keep_cells]),
  row.names  = keep_cells,
  stringsAsFactors = FALSE
)

# Key modification: wrap as SummarizedExperiment
se_nn <- SummarizedExperiment(
  assays  = list(counts = expr_mat[, keep_cells]),
  colData = meta_nn[keep_cells, ]
)

## ---- 2. Select gene sets ----
var_genes <- variableGenes(dat = se_nn, exp_labels = se_nn$study_id)
message("MetaNeighbor gene count: ", length(var_genes))

## ---- 3. Run MetaNeighborUS (genesets as list; dat = se_nn) ----
set.seed(42)
celltype_NV <- MetaNeighborUS(
  var_genes  = var_genes,          # Pass gene vector directly, no list()
  dat        = se_nn,
  study_id   = se_nn$study_id,
  cell_type  = se_nn$cell_label
  # fast_version = TRUE can accelerate (enable if memory allows)
)

In [ ]:
dim(celltype_NV)
head(rownames(celltype_NV))
head(colnames(celltype_NV))

In [ ]:
## ---- 3. Orientation: rows = macaque, cols = mouse ----
parse_name <- function(x) {
  parts <- do.call(rbind, strsplit(x, "\\|"))
  data.frame(study = parts[, 1], label = parts[, 2],
             stringsAsFactors = FALSE)
}

rn <- parse_name(rownames(celltype_NV))
cn <- parse_name(colnames(celltype_NV))

mac_rows <- rownames(celltype_NV)[rn$study == "macaque"]
mus_cols <- colnames(celltype_NV)[cn$study == "mouse"]

NV <- celltype_NV[mac_rows, mus_cols, drop = FALSE]
rownames(NV) <- sub("^macaque\\|", "", mac_rows)
colnames(NV) <- sub("^mouse\\|",   "", mus_cols)

saveRDS(NV, paste0(out_dir, "celltype_NV_MetaNeighbor.rds"))

## ---- 4. Heatmap + best match table (original logic unchanged) ----
best_match_nn <- data.frame(
  macaque_subtype = rownames(NV),
  best_mouse      = colnames(NV)[apply(NV, 1, which.max)],
  AUROC           = round(apply(NV, 1, max), 3)
)
print(best_match_nn)
write.csv(best_match_nn, paste0(out_dir, "MetaNeighbor_best_match.csv"),
          row.names = FALSE)

In [ ]:
# ---- 4. Heatmap (rows = macaque / cols = mouse) ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

ht <- Heatmap(
  NV,
  name = "AUROC",
  col  = colorRamp2(c(0.4, 0.5, 0.75, 1),
                    c("#3778AE", "#F7F7F7", "#F4A582", "#B2182B")),
  cluster_rows = TRUE,  cluster_columns = TRUE,
  clustering_method_rows = "ward.D2", clustering_method_columns = "ward.D2",
  row_names_side = "left",
  row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
  width = unit(7, "cm"), height = unit(8, "cm"),
  column_title = "mouse", row_title = "macaque"
)

png(paste0(out_dir, "MetaNeighbor_macaqueVSmouse.png"),
    width = 9, height = 8.5, units = "in", res = 300)
draw(ht, heatmap_legend_side = "right")
dev.off()

# Best match per row (AUROC: >0.75 = highly conserved)
best_match_nn <- data.frame(
  macaque_subtype = sub("^macaque\\|", "", rownames(NV)),
  best_mouse      = sub("^mouse\\|", "", colnames(NV)[apply(NV, 1, which.max)]),
  AUROC           = round(apply(NV, 1, max), 3)
)
print(best_match_nn)
write.csv(best_match_nn, paste0(out_dir, "MetaNeighbor_best_match.csv"),
          row.names = FALSE)

In [ ]:
ht <- Heatmap(
  NV,
  name = "AUROC",
  col  = colorRamp2(c(0.4, 0.5, 0.75, 1),
                    c("#3778AE", "#F7F7F7", "#F4A582", "#B2182B")),
  cluster_rows = TRUE,  cluster_columns = TRUE,
  clustering_method_rows = "ward.D2", clustering_method_columns = "ward.D2",
  row_names_side = "left",
  row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
  width = unit(7, "cm"), height = unit(8, "cm"),
  column_title = "mouse", row_title = "macaque",

  # ★ Write values into cells
  cell_fun = function(j, i, x, y, width, height, fill) {
    v <- NV[i, j]
    grid.text(sprintf("%.2f", v), x, y,
              gp = gpar(fontsize = 6.5,
                        # White text on dark bg, black text on light bg, ensure readability
                        col = ifelse(v > 0.8, "white", "black")))
  }
)

png(paste0(out_dir, "MetaNeighbor_macaqueVSmouse.png"),
    width = 9, height = 8.5, units = "in", res = 300)
draw(ht, heatmap_legend_side = "right")
dev.off()

In [ ]:
#  Violin plots comparing expression of each gene between the two species

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)

# classFiltSeurat <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/crossSpecies_final_5layers.rds")
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 1. Specify genes =====
top_genes <- c("PENK", "TAC1", "NXPH2", "RIMS2", "ROBO2", "CAPS2", "RUFY3")

## Check whether these genes are all present in the object
## (cross-species data may have gene-name matching issues)
missing_genes <- setdiff(top_genes, rownames(classFiltSeurat))
if (length(missing_genes) > 0) {
  cat("⚠️ These genes are not in the object:", paste(missing_genes, collapse = ", "), "\n")
}

cat("Genes:", paste(top_genes, collapse = ", "), "\n")

## ===== 2. Extract once + convert to long format =====
expr <- FetchData(classFiltSeurat, vars = c("species", top_genes))
expr_long <- expr %>%
  pivot_longer(cols = -species, names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = top_genes),   # order by the given sequence
         species = factor(species, levels = c("macaque", "mouse")))

## ===== 3. Plot =====
p <- ggplot(expr_long, aes(x = species, y = Expression, fill = species)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_boxplot(width = 0.15, outlier.size = 0.15, alpha = 0.7, color = "grey30") +
  facet_wrap(~ gene, nrow = 1, scales = "free_y") +
  scale_fill_manual(values = c("macaque" = "#EB6368", "mouse" = "#82ADD0")) +
  labs(title = "PENK / TAC1 / NXPH2: macaque vs mouse",
       x = NULL, y = "Normalized Expression") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1))

print(p)
ggsave("/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/VlnPlot_PENK_TAC1_NXPH2_macaque_vs_mouse.png",
       p, width = 10, height = 5, dpi = 300)  

In [ ]:
# 1. Check what .rds files are actually in the target directory
#    (watch out for case, spaces, and extensions)
list.files("/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/第五步数据保存907/",
           pattern = "\\.rds$")

# 2. If the above returns an empty character vector, search one level up
list.files("/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/",
           pattern = "crossSpecies.*\\.rds$", recursive = TRUE)

# 3. Last-resort full-disk fallback (in case the mount path has changed)
list.files("/mnt/90-connectome", pattern = "crossSpecies_CCA_dims50_mac5HT\\.rds",
           recursive = TRUE, full.names = TRUE)

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)

classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 1. Specify genes =====
top_genes <- c("PENK", "TAC1", "NXPH2", "RIMS2", "ROBO2", "CAPS2", "RUFY3")

missing_genes <- setdiff(top_genes, rownames(classFiltSeurat))
if (length(missing_genes) > 0) {
  cat("Warning: These genes are not in the object:", paste(missing_genes, collapse = ", "), "\n")
}

## ===== 2. Extract once + pivot long (also donor and cell type) =====
expr <- FetchData(classFiltSeurat,
                  vars = c("species", "donor", "subtype_recl", top_genes))

# Check actual donor and cell type values (also check NA)
table(expr$donor, useNA = "ifany")
table(expr$subtype_recl, useNA = "ifany")

expr_long <- expr %>%
  pivot_longer(cols = all_of(top_genes),
               names_to = "gene", values_to = "Expression") %>%
  mutate(gene   = factor(gene, levels = top_genes),
         donor  = factor(donor),
         celltype = factor(subtype_recl))

## ===== 3. Plot: x = cell type, fill = donor, facet = gene =====
p <- ggplot(expr_long, aes(x = celltype, y = Expression, fill = donor)) +
  geom_violin(scale = "width", trim = TRUE, position = position_dodge(width = 0.9)) +
  geom_boxplot(width = 0.12, outlier.size = 0, alpha = 0.6, color = "grey30",
               position = position_dodge(width = 0.9)) +
  facet_wrap(~ gene, nrow = 2, scales = "free_y") +
  labs(title = "Expression across cell types and donors",
       x = NULL, y = "Normalized Expression", fill = "Donor") +
  theme_bw(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold", size = 11),
        legend.position = "right")

print(p)
ggsave("./output/VlnPlot_topGenes_byDonor_celltype.png",
       p, width = 16, height = 9, dpi = 300)